In [ ]:
import os
import openai
from openai import OpenAI
from sklearn.cluster import KMeans
from utils import cosine_similarity

# initialize openai
os.environ['OPENAI_API_KEY']= ""
openai.api_key = os.environ["OPENAI_API_KEY"]

## <font color=yellow>1. 사용자 의도 파악</font>
- 사용자의 예상되는 질문을 미리 넣어놓음
- ml과 유사한 문장이면, 그에 따른 후속 함수를 나오게하는 그런 단계

In [2]:
politics = ["What are the key policies of the main political parties in the upcoming election?",
            "Who do you vote for the next presedent?",
            "I love the current Democratic Party.",
            "What is your opinion on the president's current political move?",
            "I love politics. Don't you?"]

ml = ["How does supervised learning differ from unsupervised learning in machine learning models?",
      "What are the ethical considerations of using machine learning in predictive policing?",
    "How do neural networks mimic the human brain in processing data and recognizing patterns?",
    "What are some examples of natural language processing?",
    "Can you describe how machine learning is being utilized in personalized medicine and healthcare?"]

In [3]:
def create_embeddings(txt_list):
    client = OpenAI()

    response = client.embeddings.create(
        input=txt_list,
        model="text-embedding-3-small",
        )
    responses = [r.embedding for r in response.data]
    return responses

In [5]:
embeddings = politics + ml
emb = create_embeddings(embeddings)

## <font color=yellow>2. Clustering 활용</font>

In [6]:
n_clusters = 2
kmeans = KMeans(n_clusters=n_clusters)
clusters = kmeans.fit_predict(emb)

In [7]:
clusters

array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

- 앞 쪽 5개는 politics와 관련된 문장들 
- 그래서 0 으로 묶인것임

In [8]:
input_sentence = "I would like to have a talk about politics."
sent_emb = create_embeddings([input_sentence])

In [9]:
kmeans.predict(sent_emb)

array([0])

politics 관련 질문을 해서 0 그룹에 들어감

In [10]:
input_sentence = "Tell me about machine learning."
sent_emb = create_embeddings([input_sentence])

In [11]:
kmeans.predict(sent_emb)

array([1])

ml 관련 질문을 해서 1 그룹에 들어감

## <font color=yellow>3. Silmilarity Search 활용</font>

In [12]:
politics_emb = create_embeddings(politics)
ml_emb = create_embeddings(ml)

In [ ]:
def route_selection(emb_list, query_emb, threshold=0.5):
    cos_sim = [cosine_similarity(i, query_emb) for i in emb_list]

    threshold_filtered = [i for i in cos_sim if i > threshold]

    if len(threshold_filtered) > 0:
        return True
    else:
        return False

- 기존 embeddings이랑 사용자가 새롭게 넣어준 embeddings 각각을 cosine similarity 해줌
- 특정 벡터끼리 threshold를 넘으면 사용자가 제공한 input이 미리 넣어준 politics 질문과 유사한게 있기 때문에 True를 내리게 됨 

In [14]:
input_sentence = "I would like to have a talk about politics."
sent_emb = create_embeddings([input_sentence])

print(f"{route_selection(politics_emb, sent_emb[0])} for politics, {route_selection(ml_emb, sent_emb[0])} for machine learning")

True for politics, False for machine learning


In [15]:
input_sentence = "How is the weather today?"
sent_emb = create_embeddings([input_sentence])

print(f"{route_selection(politics_emb, sent_emb[0])} for politics, {route_selection(ml_emb, sent_emb[0])} for machine learning")

False for politics, False for machine learning


In [16]:
input_sentence = "What is the best way to learn machine learning?"
sent_emb = create_embeddings([input_sentence])

print(f"{route_selection(politics_emb, sent_emb[0])} for politics, {route_selection(ml_emb, sent_emb[0], threshold=0.4)} for machine learning")

False for politics, True for machine learning


이런 임베딩 유사도를 활용해서 사용자의 목적을 파악하고, 이후의 function을 실행함으로써, 유저가 원하는 결과를 제공해줄 수 있음
- Embedding을 활용하기 때문에 최소한의 input을 활용하여 clustering이 가능해짐 <br>
##### __=> 사용자의 목적을 파악하여, 각 목적에 맞는 function 실행 가능__ (guardrails 또는 semantic router)

## <font color=yellow>4. 자주 묻는 질문 리스트</font>
1. 동일한 방식으로 자주 묻는 질문을 카테고리 별로 저장
2. threshold를 정해서 유사한 질문 search
3. 유사한 질문과 연결된 정보 제공
- 챗봇에 보면 자주 묻는 질문들이 있음

In [17]:
password_reset = ["What steps should I take to recover my account without access to my registered email?",
                  "Is there a way to authenticate my identity for password reset without security questions?",
                  "How can I reset my password?"]
service_request = ["Are there any special offers or discounts currently available?",
                   "How can I compare the different plans to find one that suits my needs?",
                   "Where can I see user reviews or testimonials about your services?"]